# **Treinando um MultiLayer Perceptron**
<font size=3>

Vamos desenvolver um primeiro modelo simples de *deep learning* do tipo **_multilayer perceptron_** (MPL) para uma tarefa de regressão. Aqui, iremos aproveitar nossos estudos prévios de pré-processamento de dados, **modelar** uma *rede neural* e salvar este modelo.


### **_Deep Learning workflow:_**
<font size=3>

Para *modelar* e avaliar corretamente uma rede neural, devemos seguir os seguintes passos de **fluxo de trabalho**:
1. **Importação e análise exploratória dos dados**;
2. **Pré-processamento dos dados**;
3. **Modelagem e compilação do modelo**;
4. **Treinamento validativo**;
5. **Treinamento final**;
6. **Avaliação do modelo**;
7. **Salvar o modelo**.


In [1]:
#from google.colab import drive
#drive.mount('/content/drive/')

In [2]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

### **1. Importação e análise exploratória dos dados:**
<font size=3>

Utilizaremos o conjunto de dados [stsb_multi_mt](https://huggingface.co/datasets/PhilipMay/stsb_multi_mt) do **Hugging Face** que é uma versão multilíngue do benchmark *Semantic Textual Similarity Benchmark* (STS-B), na qual **contém pares de sentenças** em vários idiomas, **acompanhados de uma pontuação contínua de similaridade** que varia de 0 (sem relação) a 5 (significado idêntico). Cada exemplo consiste em duas sentenças e uma nota atribuída por humanos indicando o grau de similaridade semântica entre elas.


In [3]:
# importando os dados:
df = pd.read_csv("./dataset/similarity.csv")

print(df.shape)

df.head()

(8628, 3)


,sentence1,sentence2,similarity_score
0,A plane is taking off.,An air plane is taking off.,5.00
1,A man is playing a large flute.,A man is playing a flute.,3.80
2,A man is spreading shreded cheese on a pizza.,A man is spreading shredded cheese on an uncoo...,3.80
3,Three men are playing chess.,Two men are playing chess.,2.60
4,A man is playing the cello.,A man seated is playing the cello.,4.25


### **2. Pré-processamento dos dados:**

In [4]:
# definindo a pontuação de similaridade:
y = df['similarity_score'].to_numpy(dtype='float32')

# unindo as sentenças:
X = df.apply(lambda x: x["sentence1"] + " [SEP] " + x["sentence2"], axis=1).tolist()

# definindo preprocessador para a vetorização:
def Preprocessor(text):

    text = text.lower()

    # remove todos os alfanuméricos execeto colchetes:
    text = re.sub(r"[^\w\s\[\]]", "", text)

    # remove dígitos em sequência:
    text = re.sub(r"\d+", "", text)

    return text

# vetorização:
vectorizer = TfidfVectorizer(preprocessor=Preprocessor, min_df=5, max_df=0.95)
vecs = vectorizer.fit_transform(X)
vocab = vectorizer.get_feature_names_out()

print("Vocabulary size =", len(vocab))
vocab[:10], vocab[-10:]

Vocabulary size = 3260


(array(['able', 'abortion', 'about', 'above', 'abroad', 'absolutely',
        'abuse', 'academy', 'accept', 'accepted'], dtype=object),
 array(['yourself', 'youth', 'youve', 'yuan', 'zealand', 'zero',
        'zimbabwe', 'zimmerman', 'zombies', 'zone'], dtype=object))

In [5]:
# redução de dimensionalidade:
svd = TruncatedSVD(n_components=200)
X = svd.fit_transform(vecs)

# normalização:
print(f"X: min = {X.min():.2f}, max = {X.max():.2f}")
print(f"y: min = {y.min()}, max = {y.max()}")

y /= y.max()

print(f"y: min = {y.min()}, max = {y.max()} (normalizado)\n")

# embaralhamento dos dados:
N_samples, input_size = X.shape

i = np.random.permutation(N_samples)

X = X[i]
y = y[i]

print(f"X:{X.shape}, y:{y.shape}")

X: min = -0.57, max = 0.70
y: min = 0.0, max = 5.0
y: min = 0.0, max = 1.0 (normalizado)

X:(8628, 200), y:(8628,)


In [6]:
# dividindo os dados entre trainamento, validação e teste:
N_train = int(0.7*N_samples)
N_val = int(0.2*N_samples)

X_train = X[:N_train]
X_val = X[N_train:N_train + N_val]
X_test = X[N_train + N_val:]

print(f"X-tran:{X_train.shape}, X-val:{X_val.shape}, X-test:{X_test.shape}")

y_train = y[:N_train]
y_val = y[N_train:N_train + N_val]
y_test = y[N_train + N_val:]

print(f"y-tran:{y_train.shape}, y-val:{y_val.shape}, y-test:{y_test.shape}")

X-tran:(6039, 200), X-val:(1725, 200), X-test:(864, 200)
y-tran:(6039,), y-val:(1725,), y-test:(864,)


### **3. Modelagem e compilação do modelo:**
<font size=3>

Para criar uma **rede neural**, iremos usar a biblioteca [Keras](https://keras.io/guides/functional_api/) um API do *framework* [TensorFlow](https://www.tensorflow.org/learn?hl=pt-br).

In [8]:
import keras
from keras import layers

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
x_in = keras.Input(shape=(input_size,))

x = layers.Dense(50, activation="relu")(x_in)

x_out = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs=x_in, outputs=x_out)

model.summary()

<font size=3>
<br>
    
Agora, iremos definir o **otimizador**, **função de custo**, e a **métrica** para os dados de validação.

In [ ]:
model.compile(optimizer="SGD", loss="mse", metrics=['mae'])

### **4. Treinamento validativo:**
<font size=3>

Este é o momento que validamos a modelagem da rede neural do passo anterior. Verificamos as curvas de **treinamento** e **validação** a fim de identificar se nosso modelo está **bem ajustado**, ou sofre de **_overfitting_** ou **_underfitting_**. Caso a validação não mostre um bom desempenho, devemos voltar ao passo anterior para aprimorar a rede neural — por exemplo, alterando o número de neurônios e camadas (*hiperparâmetros*) — e **reiniciar e executar todo o código**.

In [ ]:
final_training = False

n_epochs = 20
batch_saize = 16

if not final_training:

    # treinamento validativo:
    report = model.fit(x=X_train, y=y_train, validation_data=[X_val, y_val],
                       epochs=n_epochs, batch_size=batch_saize)

    # plotando as métricas ao longo das épocas:
    loss = report.history['loss']
    mae = report.history['mae']

    val_loss = report.history['val_loss']
    val_mae = report.history['val_mae']

    epochs = np.linspace(1, len(loss), len(loss))

    fig, ax = plt.subplots(1, 2, figsize=(16, 5))

    ax[0].plot(epochs, loss, label='MSE')
    ax[0].plot(epochs, val_loss, label='val-MSE')

    ax[1].plot(epochs, mae, label='MAE')
    ax[1].plot(epochs, val_mae, label='val-MAE')

    for i in range(2):
        ax[i].set_xlabel("Epochs")
        ax[i].legend()
        ax[i].grid()

    plt.show()


### **5. Treinamento final:**
<font size=3>

Com a nossa rede bem modelada, agora, faremos o treinamento do modelo final, *juntando* os dados de treinamento e validação.

In [ ]:
if final_training:

    # treinamento do modelo final:
    report = model.fit(x=np.concatenate([X_train, X_val]),
                       y=np.concatenate([y_train, y_val]),
                       epochs=n_epochs, batch_size=batch_saize)

    # plotando as métricas ao longo das épocas:
    loss = report.history['loss']
    mae = report.history['mae']

    epochs = np.linspace(1, len(loss), len(loss))

    fig, ax = plt.subplots(1, 2, figsize=(16, 5))

    ax[0].plot(epochs, loss, c='tab:blue')
    ax[1].plot(epochs, mae, c='tab:orange')

    ax[0].set_ylabel('MSE')
    ax[1].set_ylabel('ME')

    for i in range(2):
        ax[i].set_xlabel("Epochs")
        ax[i].grid()

    plt.show()


### **6. Avaliação do modelo:**
<font size=3>

Avaliando o modelo com os dados de teste.

In [ ]:
mse, mae = model.evaluate(x=X_test, y=y_test, batch_size=batch_saize)

print(f"\nMSE = {mse:.3f}, MAE = {mae:.3f}\n")

In [ ]:
y_pred = model.predict(X_test)

print("\ny-pred | y-test")
print("---------------")

for p, y in zip(y_pred[:10], y_test[:10]):

    print(f" {p[0]:.2f}  |  {y:.2f}")


### **7. Salvar o modelo (_opcional_):**
<font size=3>

Podemos, agora, salvar a arquitetura do modelo em um arquivo $\texttt{.py}$, como uma função ou classe.

<font size=2>
    
```python
def MyModel(input_size=200):
    
    x_in = keras.Input(shape=(input_size,))
    
    x = layers.Dense(50, activation="relu")(x_in)
    
    x_out = layers.Dense(1, activation="sigmoid")(x)
    
    return keras.Model(inputs=x_in, outputs=x_out)

```

<br>


In [ ]:
# salvando os parâmetros (weights, bias) do modelo:

model.save_weights("/content/drive/MyDrive/IMD1107 - Processamento de Linguagem Natural/3-unidade/weights/model.weights.h5")

In [ ]:
# salvando (x, y)-test arrays para futuras avaliações:
np.save("/content/drive/MyDrive/IMD1107 - Processamento de Linguagem Natural/3-unidade/dataset/x_test.npy", X_test)
np.save("/content/drive/MyDrive/IMD1107 - Processamento de Linguagem Natural/3-unidade/dataset/y_test.npy", y_test)

## **Referências:**
<font size=3>
    
 - **(3.4):** F. Chollet, *Deep Learning with Python*. Manning, 2018.
   